# Week 3 Lab — SOLUTIONS — Multiple Regression and the APT

**MANG2074 Financial Econometrics 1**

**Objectives**

- Transform macro variables into "news" (changes/growth rates) before regression.
- Estimate and interpret a multiple regression (APT-style factor model).
- Compute and interpret $R^2$ and adjusted $R^2$.
- Conduct joint hypothesis tests with the F-statistic, both via `f_test` and manually from restricted/unrestricted RSS.

**Data**

`../data/macro.csv` — monthly, 1986–2018: `MICROSOFT`, `SANDP` (prices), `CPI`, `INDPRO`, `M1SUPPLY`, `CCREDIT`, `BMINUSA` (Baa−Aaa credit spread), `USTB3M`, `USTB10Y` (% p.a. yields).


## Task 1 — Construct the variables

In [1]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
import warnings
warnings.filterwarnings('ignore')

raw = pd.read_csv('../data/macro.csv', index_col=0, parse_dates=True)

def logdiff(x):
    """100 * log difference = % continuously compounded growth."""
    return 100 * np.log(x / x.shift(1))

data = pd.DataFrame({
    'ermsoft':    logdiff(raw['MICROSOFT']) - raw['USTB3M'] / 12,
    'ersandp':    logdiff(raw['SANDP']) - raw['USTB3M'] / 12,
    'dprod':      raw['INDPRO'].diff(),
    'dcredit':    raw['CCREDIT'].diff(),
    'dinflation': logdiff(raw['CPI']).diff(),
    'dmoney':     raw['M1SUPPLY'].diff(),
    'dspread':    raw['BMINUSA'].diff(),
    'rterm':      (raw['USTB10Y'] - raw['USTB3M']).diff(),
}).dropna()

data.head()

,ermsoft,ersandp,dprod,dcredit,dinflation,dmoney,dspread,rterm
Date,,,,,,,,
1986-05-01,7.655834,4.373351,0.1196,7.5483,0.459855,-1.3,-0.20,0.32
1986-06-01,-13.479167,0.867757,-0.1891,5.9758,0.273590,17.1,0.01,0.02
1986-07-01,-8.099084,-6.547514,0.3137,5.7173,-0.549452,10.6,0.07,-0.10
1986-08-01,-0.474167,6.403095,-0.0748,6.9043,0.182482,5.0,0.18,0.18
1986-09-01,-1.326843,-9.376901,0.1135,9.3754,0.272271,6.1,-0.15,0.62


**What to interpret.** Each macro factor is now a *change*: under (weak-form) efficient pricing only new information should move returns, and changes are a crude proxy for news. Differencing also removes the trends in CPI, credit and M1, avoiding a regression that mixes stationary returns with trending levels. We lose two observations (one to the log-difference, one more to `dinflation`, handled by `dropna()`).

## Task 2 — Estimate the full APT-style model

In [2]:
formula = 'ermsoft ~ ersandp + dprod + dcredit + dinflation + dmoney + dspread + rterm'
results = smf.ols(formula, data).fit()
print(results.summary())


                            OLS Regression Results                            
Dep. Variable:                ermsoft   R-squared:                       0.345
Model:                            OLS   Adj. R-squared:                  0.333
Method:                 Least Squares   F-statistic:                     28.24
Date:                Thu, 11 Jun 2026   Prob (F-statistic):           3.52e-31
Time:                        01:13:04   Log-Likelihood:                -1328.3
No. Observations:                 383   AIC:                             2673.
Df Residuals:                     375   BIC:                             2704.
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.3260      0.475      2.789      0.0

**What to interpret.** The market factor `ersandp` has a t-ratio above 10 with a beta around 1.3 — Microsoft co-moves strongly with, and slightly amplifies, the market. Among the macro factors only the term-structure change `rterm` is individually significant at 5% (positive, t ≈ 2.8); the inflation surprise `dinflation` is borderline (significant at 10% only), and `dprod`, `dcredit`, `dmoney`, `dspread` are clearly insignificant. $R^2 \approx 0.34$ — typical for monthly single-stock regressions; most variation is idiosyncratic.

## Task 3 — Adjusted R² by hand

In [3]:
T = results.nobs
k = len(results.params)          # 8 parameters incl. intercept
r2 = results.rsquared
r2_adj_hand = 1 - (T - 1) / (T - k) * (1 - r2)
print(f"T = {T:.0f}, k = {k}")
print(f"R2                 = {r2:.6f}")
print(f"adj R2 (by hand)   = {r2_adj_hand:.6f}")
print(f"adj R2 (statsmodels) = {results.rsquared_adj:.6f}")


T = 383, k = 8
R2                 = 0.345205
adj R2 (by hand)   = 0.332982
adj R2 (statsmodels) = 0.332982


**What to interpret.** The two values match. $\bar R^2$ sits below $R^2$ because we are "paying" degrees-of-freedom rent for seven regressors, several of which contribute almost nothing — exactly the over-fitting penalty $\bar R^2$ is designed to impose.

## Task 4 — Joint F-test on the macro factors

In [4]:
hypotheses = 'dprod = dcredit = dmoney = dspread = 0'
f_res = results.f_test(hypotheses)
print(f_res)


<F test: F=0.4138785595154818, p=0.7986453783446849, df_denom=375, df_num=4>


**What to interpret.** $H_0$: the four coefficients are all zero (the factors are jointly unpriced); under $H_0$ the statistic is $F(4, T-8)$. The p-value is well above 0.05, so we **fail to reject**: these four macro factors add no significant joint explanatory power for Microsoft once the market, inflation surprises and the term-structure are in the model. Note this is the joint version of what the individual t-ratios hinted — but the F-test is the correct tool, since four marginal t-tests do not control the joint size.

## Task 5 — Replicate the F-test manually

In [5]:
from scipy import stats

# The restricted model imposes the four zero restrictions, i.e. it drops
# dprod, dcredit, dmoney and dspread but keeps everything else:
restricted = smf.ols('ermsoft ~ ersandp + dinflation + rterm', data).fit()

URSS = results.ssr
RRSS = restricted.ssr
m = 4                       # number of restrictions
T = results.nobs
k = len(results.params)     # parameters in the UNrestricted model

F = ((RRSS - URSS) / m) / (URSS / (T - k))
pval = stats.f.sf(F, m, T - k)
print(f"URSS = {URSS:.2f}, RRSS = {RRSS:.2f}")
print(f"F (by hand) = {F:.4f}, p-value = {pval:.4f}")


URSS = 23078.11, RRSS = 23179.99
F (by hand) = 0.4139, p-value = 0.7986


**What to interpret.** The hand-computed F and p-value coincide with `f_test` (up to rounding). The mechanics make the test transparent: imposing the four zero restrictions raises the RSS only slightly, so the evidence against $H_0$ is weak. (Note the restricted model must drop *exactly* the four variables under test and keep everything else.)

## Task 6 — A trimmed model

In [6]:
trimmed = smf.ols('ermsoft ~ ersandp + dinflation + rterm', data).fit()
print(trimmed.summary())

comp = pd.DataFrame({'full': [results.rsquared_adj, results.aic, results.bic],
                     'trimmed': [trimmed.rsquared_adj, trimmed.aic, trimmed.bic]},
                    index=['adj R2', 'AIC', 'BIC'])
print(comp.round(3))


                            OLS Regression Results                            
Dep. Variable:                ermsoft   R-squared:                       0.342
Model:                            OLS   Adj. R-squared:                  0.337
Method:                 Least Squares   F-statistic:                     65.75
Date:                Thu, 11 Jun 2026   Prob (F-statistic):           2.99e-34
Time:                        01:13:04   Log-Likelihood:                -1329.2
No. Observations:                 383   AIC:                             2666.
Df Residuals:                     379   BIC:                             2682.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      1.0209      0.401      2.546      0.0

**What to interpret.** The trimmed model has essentially the same adjusted $R^2$ and *lower* AIC and BIC (information criteria reward fit but penalise parameters; lower = better). The retained coefficients barely move, confirming the dropped factors were dead weight. We would report the trimmed model — but honestly note that the specification search was guided by the data (a caveat for inference).

## Task 7 — Interpretation

The market factor dominates: Microsoft's excess return loads on the S&P 500 with a beta of about 1.2–1.3 and a t-ratio above 10, while the block of "real economy" factors (production, credit, money, default spread) is jointly insignificant. Only the change in the yield-curve slope adds clearly significant explanatory power, with inflation surprises borderline. For this stock over 1986–2018, then, a one-factor CAPM-style description is close to adequate, and the APT's extra macro factors earn their keep only marginally. Two caveats: (i) monthly changes are a noisy proxy for genuine macro *news* — surveys of expectations would be sharper; (ii) factor significance can be sample-dependent and unstable across sub-periods, which we test next week (Chow test).